In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Load the datasets
try:
    perlintasan_df = pd.read_csv('data/Perlintasan_Pelayaran.csv')
    pdrb_df = pd.read_csv('data/PDRB_Forecast_2025-2029.csv')
    print("Data loaded successfully!")
except FileNotFoundError as e:
    print(f"Error loading data: {e}")


Data loaded successfully!


In [19]:
print(f"PDRB Data Shape: {pdrb_df.shape}")
print(f"Years available: {pdrb_df['Tahun'].unique()}")
display(pdrb_df.head())

PDRB Data Shape: (425, 27)
Years available: [2025 2026 2027 2028 2029]


,KabKota,Tahun,Pengeluaran Konsumsi Rumah Tangga,Pengeluaran Konsumsi LNPRT,Pengeluaran Konsumsi Pemerintah,Pembentukan Modal Tetap Bruto,Perubahan Inventori,Net Ekspor,PDRB,"A Pertanian, Kehutanan, dan Perikanan",B Pertambangan dan Penggalian,C Industri Pengolahan,D Pengadaan Listrik dan Gas,"E Pengadaan Air; Pengelolaan Sampah, Limbah, dan Daur Ulang",F Konstruksi,G Perdagangan Besar dan Eceran; Reparasi Mobil dan Sepeda Motor,H Transportasi dan Pergudangan,I Penyediaan Akomodasi dan Makan Minum,J Informasi dan Komunikasi,K Jasa Keuangan dan Asuransi,L Real Estat,"M,N Jasa Perusahaan","O Administrasi Pemerintahan, Pertahanan, dan Jaminan Sosial Wajib",P Jasa Pendidikan,Q Jasa Kesehatan dan Kegiatan Sosial,"R,S,T,U Jasa Lainnya",Produk Domestik Bruto
0,Aceh Barat,2025,5258.295136,300.888877,1581.927298,5054.058798,100.455015,2311.267886,14075.662817,4145.675881,4188.021270,264.170210,13.077441,2.924969,1232.025907,1794.869029,566.819250,170.356330,316.550289,208.974979,302.130639,48.927233,1009.708997,237.366683,347.133480,74.583648,14075.662817
1,Aceh Selatan,2025,4487.590527,140.246195,1386.360943,2549.640094,-9.867925,-1171.623307,7341.960022,2264.978978,202.562500,228.474075,5.229675,2.113116,1176.986340,1149.220730,322.306286,39.445856,333.580629,175.647865,260.338648,46.518626,829.013571,227.603363,147.217290,96.073205,7341.960022
2,Aceh Singkil,2025,2041.232690,62.173155,1037.682697,958.882479,0.234819,-679.628304,3518.665440,1108.595809,37.469726,232.336425,5.493823,1.530909,333.557947,424.298053,179.174779,44.887131,119.234773,63.538248,95.184615,20.208330,485.510504,98.721777,160.858855,39.303352,3518.665440
3,Alor,2025,2876.937922,133.082672,1401.825951,1694.440683,52.992516,-2155.923706,3884.807125,1265.012017,49.811915,58.514014,5.266051,3.576068,417.230036,534.796541,219.478697,18.994371,229.201459,288.027121,69.317870,15.280708,653.845030,118.898407,28.139471,17.124339,3884.807125
4,Banggai,2025,13887.265109,826.884901,3775.910777,7896.064255,335.819118,13173.597374,40319.168459,7317.768140,9179.033838,11754.382215,6.453525,19.014874,3379.455449,2153.355981,1150.600916,151.707533,935.793282,722.834912,592.802909,34.202006,1447.160917,873.528941,299.092750,174.838210,40319.168459


In [20]:
# Get unique cities from Perlintasan
cities_origin = perlintasan_df['Kab/Kota Keberangkatan'].unique()
cities_dest = perlintasan_df['Kab/Kota Tujuan'].unique()
all_route_cities = set(cities_origin) | set(cities_dest)

# Get unique cities from PDRB (using the raw dataframe)
pdrb_cities = set(pdrb_df['KabKota'].unique())

print(f"Number of cities in routes: {len(all_route_cities)}")
print(f"Number of cities in PDRB data: {len(pdrb_cities)}")

Number of cities in routes: 84
Number of cities in PDRB data: 85


In [ ]:
# --- Merge Logic for Yearly Analysis ---

# 1. Merge PDRB data for Origin
# This will expand the dataframe: Each route will now have rows for each year present in pdrb_df
route_yearly_df = pd.merge(
    perlintasan_df,
    pdrb_df,
    left_on='Kab/Kota Keberangkatan',
    right_on='KabKota',
    how='left'
).rename(columns=lambda x: x + '_Origin' if x in pdrb_df.columns and x != 'KabKota' and x != 'Tahun' else x)

# Drop the redundant KabKota column from merge
route_yearly_df = route_yearly_df.drop(columns=['KabKota'])

# 2. Merge PDRB data for Destination
# We must merge on 'Kab/Kota Tujuan' AND 'Tahun' to align the years
route_yearly_df = pd.merge(
    route_yearly_df,
    pdrb_df,
    left_on=['Kab/Kota Tujuan', 'Tahun'],
    right_on=['KabKota', 'Tahun'],
    how='left',
    suffixes=('', '_Dest')
).rename(columns=lambda x: x + '_Dest' if x in pdrb_df.columns and x != 'KabKota' and x != 'Tahun' and not x.endswith('_Origin') else x)

# Drop the redundant KabKota column from merge
route_yearly_df = route_yearly_df.drop(columns=['KabKota'])

print("Merged Yearly DataFrame shape:", route_yearly_df.shape)
# Expected shape: (Num_Routes * 5_Years, Columns)

# --- Calculate Route Totals ---
# Identify PDRB columns (numeric columns that came from pdrb_df)
# We exclude 'Tahun' and the key columns
pdrb_columns = [col for col in pdrb_df.columns if col not in ['KabKota', 'Tahun']]

for col in pdrb_columns:
    origin_col = f"{col}_Origin"
    dest_col = f"{col}_Dest"
    route_col = f"{col}_Route"
    
    # Sum Origin and Destination values
    # Fillna(0) is important in case one city is missing data, though ideally we want complete data
    route_yearly_df[route_col] = route_yearly_df[origin_col].fillna(0) + route_yearly_df[dest_col].fillna(0)

# Display a sample
display(route_yearly_df[['Lintas', 'Tahun', 'Produk Domestik Bruto_Route']].head(10))

Merged Yearly DataFrame shape: (325, 57)


,Lintas,Jarak (Mil),Klasifikasi,Pengelola,Kab/Kota Keberangkatan,Kab/Kota Tujuan,Tahun,Pengeluaran Konsumsi Rumah Tangga_Origin,Pengeluaran Konsumsi LNPRT_Origin,Pengeluaran Konsumsi Pemerintah_Origin,Pembentukan Modal Tetap Bruto_Origin,Perubahan Inventori_Origin,Net Ekspor_Origin,PDRB_Origin,"A Pertanian, Kehutanan, dan Perikanan_Origin",B Pertambangan dan Penggalian_Origin,C Industri Pengolahan_Origin,D Pengadaan Listrik dan Gas_Origin,"E Pengadaan Air; Pengelolaan Sampah, Limbah, dan Daur Ulang_Origin",F Konstruksi_Origin,G Perdagangan Besar dan Eceran; Reparasi Mobil dan Sepeda Motor_Origin,H Transportasi dan Pergudangan_Origin,I Penyediaan Akomodasi dan Makan Minum_Origin,J Informasi dan Komunikasi_Origin,K Jasa Keuangan dan Asuransi_Origin,L Real Estat_Origin,"M,N Jasa Perusahaan_Origin","O Administrasi Pemerintahan, Pertahanan, dan Jaminan Sosial Wajib_Origin",P Jasa Pendidikan_Origin,Q Jasa Kesehatan dan Kegiatan Sosial_Origin,"R,S,T,U Jasa Lainnya_Origin",Produk Domestik Bruto_Origin,Pengeluaran Konsumsi Rumah Tangga_Dest,Pengeluaran Konsumsi LNPRT_Dest,Pengeluaran Konsumsi Pemerintah_Dest,Pembentukan Modal Tetap Bruto_Dest,Perubahan Inventori_Dest,Net Ekspor_Dest,PDRB_Dest,"A Pertanian, Kehutanan, dan Perikanan_Dest",B Pertambangan dan Penggalian_Dest,C Industri Pengolahan_Dest,D Pengadaan Listrik dan Gas_Dest,"E Pengadaan Air; Pengelolaan Sampah, Limbah, dan Daur Ulang_Dest",F Konstruksi_Dest,G Perdagangan Besar dan Eceran; Reparasi Mobil dan Sepeda Motor_Dest,H Transportasi dan Pergudangan_Dest,I Penyediaan Akomodasi dan Makan Minum_Dest,J Informasi dan Komunikasi_Dest,K Jasa Keuangan dan Asuransi_Dest,L Real Estat_Dest,"M,N Jasa Perusahaan_Dest","O Administrasi Pemerintahan, Pertahanan, dan Jaminan Sosial Wajib_Dest",P Jasa Pendidikan_Dest,Q Jasa Kesehatan dan Kegiatan Sosial_Dest,"R,S,T,U Jasa Lainnya_Dest",Produk Domestik Bruto_Dest,Pengeluaran Konsumsi Rumah Tangga_Route,Pengeluaran Konsumsi LNPRT_Route,Pengeluaran Konsumsi Pemerintah_Route,Pembentukan Modal Tetap Bruto_Route,Perubahan Inventori_Route,Net Ekspor_Route,PDRB_Route,"A Pertanian, Kehutanan, dan Perikanan_Route",B Pertambangan dan Penggalian_Route,C Industri Pengolahan_Route,D Pengadaan Listrik dan Gas_Route,"E Pengadaan Air; Pengelolaan Sampah, Limbah, dan Daur Ulang_Route",F Konstruksi_Route,G Perdagangan Besar dan Eceran; Reparasi Mobil dan Sepeda Motor_Route,H Transportasi dan Pergudangan_Route,I Penyediaan Akomodasi dan Makan Minum_Route,J Informasi dan Komunikasi_Route,K Jasa Keuangan dan Asuransi_Route,L Real Estat_Route,"M,N Jasa Perusahaan_Route","O Administrasi Pemerintahan, Pertahanan, dan Jaminan Sosial Wajib_Route",P Jasa Pendidikan_Route,Q Jasa Kesehatan dan Kegiatan Sosial_Route,"R,S,T,U Jasa Lainnya_Route",Produk Domestik Bruto_Route
0,Merak – Bakauheni,17,Antar Provinsi,ASDP,Kota Cilegon,Lampung Selatan,2025,28089.337399,86.842752,2100.154516,68875.981478,322.196603,40709.919571,148379.530738,232.187633,50.681319,79192.263073,5345.326768,254.019648,28839.263986,12366.844600,3500.958129,2297.365348,1186.128970,4521.394502,6781.169356,468.069872,1258.869415,1115.699140,1113.291622,1255.365264,148379.530738,29815.015057,1223.177226,2559.231903,18343.684921,214.728855,2615.254469,61495.877444,15236.461798,806.762309,12218.881923,69.271214,72.934073,12285.085418,8906.143438,3331.570296,641.520733,2045.856397,1083.742877,1290.343230,50.595750,1186.155304,1510.046936,352.602084,403.055039,61495.877444,57904.352456,1310.019979,4659.386419,87219.666399,536.925457,43325.174040,209875.408182,15468.649431,857.443628,91411.144996,5414.597982,326.953721,41124.349404,21272.988038,6832.528425,2938.886082,3231.985366,5605.137379,8071.512585,518.665622,2445.024720,2625.746076,1465.893707,1658.420304,209875.408182
1,Merak – Bakauheni,17,Antar Provinsi,ASDP,Kota Cilegon,Lampung Selatan,2026,29871.506323,94.671331,2143.276964,74384.296154,192.220218,40402.984073,151436.067000,256.307810,51.888731,83379.869692,4731.291584,256.227736,32341.890124,12685.1

,Lintas,Tahun,Produk Domestik Bruto_Route
0,Merak – Bakauheni,2025,209875.408182
1,Merak – Bakauheni,2026,212238.034274
2,Merak – Bakauheni,2027,224154.441072
3,Merak – Bakauheni,2028,215351.346521
4,Merak – Bakauheni,2029,220325.764205
5,Ujung – Kamal,2025,615205.348535
6,Ujung – Kamal,2026,613661.859041
7,Ujung – Kamal,2027,604920.087782
8,Ujung – Kamal,2028,583233.111073
9,Ujung – Kamal,2029,562497.578453


In [23]:
# --- Define Column Groups ---
pengeluaran_cols = [
    'Pengeluaran Konsumsi Rumah Tangga', 'Pengeluaran Konsumsi LNPRT', 
    'Pengeluaran Konsumsi Pemerintah', 'Pembentukan Modal Tetap Bruto', 
    'Perubahan Inventori', 'Net Ekspor', 'PDRB'
]

lapangan_usaha_cols = [
    "A Pertanian, Kehutanan, dan Perikanan", "B Pertambangan dan Penggalian", 
    "C Industri Pengolahan", "D Pengadaan Listrik dan Gas", 
    "E Pengadaan Air; Pengelolaan Sampah, Limbah, dan Daur Ulang", "F Konstruksi", 
    "G Perdagangan Besar dan Eceran; Reparasi Mobil dan Sepeda Motor", 
    "H Transportasi dan Pergudangan", "I Penyediaan Akomodasi dan Makan Minum", 
    "J Informasi dan Komunikasi", "K Jasa Keuangan dan Asuransi", "L Real Estat", 
    "M,N Jasa Perusahaan", "O Administrasi Pemerintahan, Pertahanan, dan Jaminan Sosial Wajib", 
    "P Jasa Pendidikan", "Q Jasa Kesehatan dan Kegiatan Sosial", 
    "R,S,T,U Jasa Lainnya", "Produk Domestik Bruto"
]

# --- Calculate Growth Rates ---
# Sort first to ensure years are in order
route_yearly_df = route_yearly_df.sort_values(by=['Lintas', 'Tahun'])

# Create a list of all route columns we want to analyze
all_target_cols = pengeluaran_cols + lapangan_usaha_cols
# Filter out columns that might not exist in the dataframe (just in case)
existing_cols = [c for c in all_target_cols if f"{c}_Route" in route_yearly_df.columns]
route_target_cols = [f"{c}_Route" for c in existing_cols]

# Calculate % Growth
# We group by 'Lintas' and apply pct_change to the numeric columns
# We'll store these in new columns with suffix '_Growth'
growth_df = route_yearly_df[['Lintas', 'Tahun'] + route_target_cols].copy()
for col in route_target_cols:
    growth_df[f"{col}_Growth"] = growth_df.groupby('Lintas')[col].pct_change() * 100

# --- Aggregate Summary (2025-2029) ---
# 1. Average Value (2025-2029)
avg_values = route_yearly_df.groupby('Lintas')[route_target_cols].mean().reset_index()

# 2. Average Growth (2026-2029, since 2025 is NaN for growth)
growth_cols = [f"{c}_Growth" for c in route_target_cols]
avg_growth = growth_df.groupby('Lintas')[growth_cols].mean().reset_index()

# Merge summaries
summary_df = pd.merge(avg_values, avg_growth, on='Lintas')

# Add Route Info (Origin/Dest) back for display
route_info = perlintasan_df[['Lintas', 'Kab/Kota Keberangkatan', 'Kab/Kota Tujuan']].drop_duplicates()
summary_df = pd.merge(route_info, summary_df, on='Lintas')

# --- Analysis Function ---
def analyze_sector(sector_name, top_n=5):
    route_col = f"{sector_name}_Route"
    growth_col = f"{sector_name}_Route_Growth"
    
    if route_col not in summary_df.columns:
        print(f"Column {route_col} not found in data.")
        return

    print(f"\n{'='*20} ANALISIS SEKTOR: {sector_name} {'='*20}")
    
    # Top by Value
    top_val = summary_df.sort_values(by=route_col, ascending=False).head(top_n)
    print(f"\nTop {top_n} Rute Berdasarkan NILAI RATA-RATA (2025-2029):")
    display(top_val[['Lintas', 'Kab/Kota Keberangkatan', 'Kab/Kota Tujuan', route_col, growth_col]])
    
    # Top by Growth
    top_growth = summary_df.sort_values(by=growth_col, ascending=False).head(top_n)
    print(f"\nTop {top_n} Rute Berdasarkan PERTUMBUHAN RATA-RATA (% per tahun):")
    display(top_growth[['Lintas', 'Kab/Kota Keberangkatan', 'Kab/Kota Tujuan', route_col, growth_col]])
    
    # The "Best" (Weighted Score: 70% Growth, 30% Value)
    # Rank by Value (1 is best)
    summary_df['Rank_Val'] = summary_df[route_col].rank(ascending=False)
    # Rank by Growth (1 is best)
    summary_df['Rank_Growth'] = summary_df[growth_col].rank(ascending=False)
    
    # NEW FORMULA: Prioritize Growth
    summary_df['Combined_Score'] = (summary_df['Rank_Growth'] * 0.7) + (summary_df['Rank_Val'] * 0.3)
    
    best_combined = summary_df.sort_values(by='Combined_Score').head(1)
    print(f"\n>>> RUTE TERBAIK (Utama: Pertumbuhan, Sekunder: Nilai) <<<")
    print(f"Rute: {best_combined['Lintas'].values[0]}")
    print(f"Asal: {best_combined['Kab/Kota Keberangkatan'].values[0]} -> Tujuan: {best_combined['Kab/Kota Tujuan'].values[0]}")
    print(f"Nilai Rata-rata: {best_combined[route_col].values[0]:,.2f}")
    print(f"Pertumbuhan Rata-rata: {best_combined[growth_col].values[0]:.2f}%")

# --- Run Analysis ---

# 1. Total PDRB (Lapangan Usaha)
analyze_sector('Produk Domestik Bruto', top_n=5)

# 2. Total PDRB (Pengeluaran)
analyze_sector('PDRB', top_n=5)


==================== ANALISIS SEKTOR: Produk Domestik Bruto ====================

Top 5 Rute Berdasarkan NILAI RATA-RATA (2025-2029):


,Lintas,Kab/Kota Keberangkatan,Kab/Kota Tujuan,Produk Domestik Bruto_Route,Produk Domestik Bruto_Route_Growth
1,Ujung – Kamal,Kota Surabaya,Bangkalan,595903.596977,-2.203947
47,Sei Selari - Bengkalis (Air Putih),Bengkalis,Bengkalis,261648.322611,0.017789
0,Merak – Bakauheni,Kota Cilegon,Lampung Selatan,216388.998851,1.280758
9,Telaga Punggur - Tj. Uban,Kota Batam,Bintan,209139.837288,-5.756234
51,Dumai – Rupat,Kota Dumai,Bengkalis,202912.856505,1.911004



Top 5 Rute Berdasarkan PERTUMBUHAN RATA-RATA (% per tahun):


,Lintas,Kab/Kota Keberangkatan,Kab/Kota Tujuan,Produk Domestik Bruto_Route,Produk Domestik Bruto_Route_Growth
43,Kota – Siantan,Kota Pontianak,Kota Pontianak,127299.289032,14.721756
62,Tj. Balai Karimun - Mengkapan,Karimun,Siak,140613.425810,5.231502
2,Ketapang – Gilimanuk,Banyuwangi,Jembrana,113202.070024,4.711373
52,Padang - Tua Pejat,Kota Padang,Kepulauan Mentawai,93743.277922,4.133591
17,Kupang – Kalabahi,Kupang,Alor,15509.184020,3.884343



>>> RUTE TERBAIK (Utama: Pertumbuhan, Sekunder: Nilai) <<<
Rute: Kota – Siantan
Asal: Kota Pontianak -> Tujuan: Kota Pontianak
Nilai Rata-rata: 127,299.29
Pertumbuhan Rata-rata: 14.72%

==================== ANALISIS SEKTOR: PDRB ====================

Top 5 Rute Berdasarkan NILAI RATA-RATA (2025-2029):


,Lintas,Kab/Kota Keberangkatan,Kab/Kota Tujuan,PDRB_Route,PDRB_Route_Growth
1,Ujung – Kamal,Kota Surabaya,Bangkalan,595903.596977,-2.203947
47,Sei Selari - Bengkalis (Air Putih),Bengkalis,Bengkalis,261648.322611,0.017789
0,Merak – Bakauheni,Kota Cilegon,Lampung Selatan,216388.998851,1.280758
9,Telaga Punggur - Tj. Uban,Kota Batam,Bintan,209139.837288,-5.756234
51,Dumai – Rupat,Kota Dumai,Bengkalis,202912.856505,1.911004



Top 5 Rute Berdasarkan PERTUMBUHAN RATA-RATA (% per tahun):


,Lintas,Kab/Kota Keberangkatan,Kab/Kota Tujuan,PDRB_Route,PDRB_Route_Growth
43,Kota – Siantan,Kota Pontianak,Kota Pontianak,127299.289032,14.721756
62,Tj. Balai Karimun - Mengkapan,Karimun,Siak,140613.425810,5.231502
2,Ketapang – Gilimanuk,Banyuwangi,Jembrana,113202.070024,4.711373
52,Padang - Tua Pejat,Kota Padang,Kepulauan Mentawai,93743.277922,4.133591
17,Kupang – Kalabahi,Kupang,Alor,15509.184020,3.884343



>>> RUTE TERBAIK (Utama: Pertumbuhan, Sekunder: Nilai) <<<
Rute: Kota – Siantan
Asal: Kota Pontianak -> Tujuan: Kota Pontianak
Nilai Rata-rata: 127,299.29
Pertumbuhan Rata-rata: 14.72%


In [17]:
# --- Summary of Best Routes for ALL Sectors ---
best_routes_summary = []

for sector in existing_cols:
    route_col = f"{sector}_Route"
    growth_col = f"{sector}_Route_Growth"
    
    # Calculate Ranks
    summary_df['Rank_Val'] = summary_df[route_col].rank(ascending=False)
    summary_df['Rank_Growth'] = summary_df[growth_col].rank(ascending=False)
    
    # NEW FORMULA: Prioritize Growth (70% Growth, 30% Value)
    summary_df['Combined_Score'] = (summary_df['Rank_Growth'] * 0.7) + (summary_df['Rank_Val'] * 0.3)
    
    best = summary_df.sort_values(by='Combined_Score').iloc[0]
    
    best_routes_summary.append({
        'Sektor': sector,
        'Rute Terbaik': best['Lintas'],
        'Nilai Rata-rata': best[route_col],
        'Pertumbuhan (%)': best[growth_col]
    })

best_routes_df = pd.DataFrame(best_routes_summary)
print("\n=== RANGKUMAN RUTE TERBAIK UNTUK SETIAP SEKTOR (Prioritas Pertumbuhan) ===")
display(best_routes_df)


=== RANGKUMAN RUTE TERBAIK UNTUK SETIAP SEKTOR (Prioritas Pertumbuhan) ===


,Sektor,Rute Terbaik,Nilai Rata-rata,Pertumbuhan (%)
0,Pengeluaran Konsumsi Rumah Tangga,Balikpapan – Taipa,69532.630729,13.964721
1,Pengeluaran Konsumsi LNPRT,Poka – Galala,1628.982972,13.575607
2,Pengeluaran Konsumsi Pemerintah,Kota – Siantan,11533.018336,7.332585
3,Pembentukan Modal Tetap Bruto,Padang - Tua Pejat,58818.084702,14.726075
4,Perubahan Inventori,Sei Selari - Bengkalis (Air Putih),336.301194,41.654416
5,Net Ekspor,Batu Licin - Tanjung Serdang,72403.185395,19.867832
6,PDRB,Kota – Siantan,127299.289032,14.721756
7,"A Pertanian, Kehutanan, dan Perikanan",Palembang (Tj. Api-Api) - (Tj. Kalian),20063.454656,9.751398
8,B Pertambangan dan Penggalian,Luwuk – Salakan,15612.980067,24.896701
9,C Industri Pengolahan,Kota – Siantan,62207.302388,64.213817
